<a href="https://colab.research.google.com/github/joterojordanufv/PracticaProgramacion/blob/main/notebooks/01_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Load

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

folder = "/content/drive/MyDrive/Data Visualization Project/"

# There is 2 sheets because of the number of rows (separated years): sheet_name=None reads both
sheets = pd.read_excel(folder + "online_retail_II.xlsx", sheet_name=None)

for name, sheet in sheets.items():
    print(name, len(sheet), "rows")

Year 2009-2010 525461 rows
Year 2010-2011 541910 rows


In [4]:
# Join the tables and create a dataframe.
df = pd.concat(sheets.values(), ignore_index=True)

# Defining columns names.
df.columns = ["invoice", "stock_code", "description", "quantity",
              "invoice_date", "price", "customer_id", "country"]

# Standarizing and converting numbers into string(text) to solve future problems and adapt to the database.
df["invoice"] = df["invoice"].astype(str)

df["stock_code"] = df["stock_code"].astype(str)

df["description"] = df["description"].astype("string")

df["country"] = df["country"].astype("string")
# Checking rows length as it says in the instructions : 1,067,371
print(len(df), "rows")

# Checking if the colums are right by showing the first 5 results
print(df.head())

1067371 rows
  invoice stock_code                          description  quantity  \
0  489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434     79323P                   PINK CHERRY LIGHTS        12   
2  489434     79323W                  WHITE CHERRY LIGHTS        12   
3  489434      22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434      21232       STRAWBERRY CERAMIC TRINKET BOX        24   

         invoice_date  price  customer_id         country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [5]:
# Save the table in Drive to keep the dataframe. Using index=False so we dont save the row numbers.
df.to_parquet(folder + "raw_combined.parquet", index=False)

# Problem Solving

In [6]:
# Summary to see type and number of data
df.info()

# Number of empty cells counter
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  string        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[ns]
 5   price         1067371 non-null  float64       
 6   customer_id   824364 non-null   float64       
 7   country       1067371 non-null  string        
dtypes: datetime64[ns](1), float64(2), int64(1), object(2), string(2)
memory usage: 65.1+ MB
invoice              0
stock_code           0
description       4382
quantity             0
invoice_date         0
price                0
customer_id     243007
country              0
dtype: int64


In [7]:
# Number of duplicated rows (Same customer, Same product, Country, etc)
print("Duplicate rows:", df.duplicated().sum())

# Number of cancelations
print("Invoices starting with C (cancellations):", df["invoice"].str.startswith("C").sum())

# Rows with negative quiantity which is imposible
print("Rows with negative quantity:", (df["quantity"] < 0).sum())

# Same iwith negative prices
print("Rows with price of 0 or less:", (df["price"] <= 0).sum())

# Count the number of empty customer_id
print("Rows without a customer number:", df["customer_id"].isna().sum())

# Know the time frame, when was the first and last order registered
print("First date:", df["invoice_date"].min())
print("Last date:", df["invoice_date"].max())

Duplicate rows: 34335
Invoices starting with C (cancellations): 19494
Rows with negative quantity: 22950
Rows with price of 0 or less: 6207
Rows without a customer number: 243007
First date: 2009-12-01 07:45:00
Last date: 2011-12-09 12:50:00


In [8]:
# Checking for strange names and objects to categorize them as "not products"
not_products = df[~df["stock_code"].str.match(r"^\d{5}")]

# Looking for most frequent strange names (15) to see if we are right deleting them
print(not_products["stock_code"].value_counts().head(15))

stock_code
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
Name: count, dtype: int64


In [9]:
# After doubting if some names were products or not we showed the whole picture.

# Counting codes not starting with 5 digits
print("Different codes that do not start with 5 digits:", not_products["stock_code"].nunique())

# Printing the whole list
print(not_products["stock_code"].value_counts().iloc[15:])

# The description of the codes we are unsure to check if they really are products
unsure = not_products[not_products["stock_code"].isin(["DCGS0058", "DCGSSGIRL", "DCGSSBOY", "PADS", "gift_0001_20", "S", "D", "M"])]
print(unsure.groupby("stock_code")["description"].first())

Different codes that do not start with 5 digits: 62
stock_code
CRUK            16
gift_0001_10    16
DCGS0076        15
TEST001         15
DCGS0003        14
gift_0001_50     8
gift_0001_40     7
B                6
DCGS0069         6
DCGS0004         5
m                5
gift_0001_80     4
DCGS0066N        4
DCGS0072         4
DCGS0068         3
DCGS0070         3
SP1002           3
gift_0001_70     3
ADJUST2          3
TEST002          2
gift_0001_60     2
DCGS0037         2
gift_0001_90     2
DCGS0062         2
DCGS0016         1
DCGS0044         1
DCGS0006         1
DCGSLBOY         1
GIFT             1
DCGS0027         1
DCGS0036         1
DCGSLGIRL        1
DCGS0053         1
DCGS0039         1
DCGS0060         1
DCGS0056         1
DCGS0059         1
DCGS0041         1
DCGS0075         1
C3               1
DCGS0055         1
DCGS0057         1
DCGS0074         1
DCGS0073         1
DCGS0071         1
DCGS0066P        1
DCGS0067         1
Name: count, dtype: int64
stock_code
D      

# Cleaning

In [10]:
# Having a first look for duplicate rows
print(df[df.duplicated(keep=False)].sort_values(["invoice", "stock_code", "invoice_date"]).head(10))

    invoice stock_code                       description  quantity  \
379  489517      21491   SET OF THREE VINTAGE GIFT WRAPS         1   
391  489517      21491   SET OF THREE VINTAGE GIFT WRAPS         1   
365  489517      21821  GLITTER STAR GARLAND WITH BELLS          1   
386  489517      21821  GLITTER STAR GARLAND WITH BELLS          1   
363  489517      21912          VINTAGE SNAKES & LADDERS         1   
371  489517      21912          VINTAGE SNAKES & LADDERS         1   
394  489517      21912          VINTAGE SNAKES & LADDERS         1   
362  489517      21913    VINTAGE SEASIDE JIGSAW PUZZLES         1   
385  489517      21913    VINTAGE SEASIDE JIGSAW PUZZLES         1   
368  489517      22130  PARTY CONE CHRISTMAS DECORATION          6   

           invoice_date  price  customer_id         country  
379 2009-12-01 11:34:00   1.95      16329.0  United Kingdom  
391 2009-12-01 11:34:00   1.95      16329.0  United Kingdom  
365 2009-12-01 11:34:00   3.75      16329.0

In [11]:
dups = df[df.duplicated()]

# Overlapping rows, they repeat because they are in both sheets
overlap = dups[(dups["invoice_date"] >= "2010-12-01") & (dups["invoice_date"] < "2010-12-10")]

print("Duplicates in total:", len(dups))
print("Of them, dated 1-9 December 2010 (the overlap):", len(overlap))
print("Duplicates on all other dates:", len(dups) - len(overlap))

Duplicates in total: 34335
Of them, dated 1-9 December 2010 (the overlap): 22844
Duplicates on all other dates: 11491


In [12]:
# The duplicates that are not overlapping
other = dups[~dups.index.isin(overlap.index)]

# Checking the rows to be ok (prices above 0)
ok = df[df["price"] > 0]
other_ok = other[other["price"] > 0]

# Calculating the money in those purchases to be aware of the results if we decide to delete them.
money_all = (ok["quantity"] * ok["price"]).sum()
money_other = (other_ok["quantity"] * other_ok["price"]).sum()

print("Doubtful duplicates:", len(other))
print("Money in them:", round(money_other, 2))
print("Share of all the money:", round(money_other / money_all * 100, 2), "%")

Doubtful duplicates: 11491
Money in them: 53006.79
Share of all the money: 0.27 %


We decided to remove 34,335  duplicate lines (3.2% of rows). 22,844 were in the days both sheets overlap (1–9 Dec 2010), so they are the same purchases or sales twice. The resting 11,491 could be some repeated scans, or similar but they have little representation in the whole picture; about 0.27% which sums up in terms of revenue in £53,007, so the choice does not change our conclusions.

In [13]:
# A list that writes down every cleaning step
steps = [["Start (both sheets)", 0, len(df)]]

# The before number of rows
before = len(df)

# drop_duplicates() removes rows identical to an earlier row (it keeps the first one). We save the result back into df.
df = df.drop_duplicates()

# Add a line to the list: before - len(df) = how many rows disappeared, len(df) = how many are left.
steps.append(["Remove duplicate rows", before - len(df), len(df)])

# Show the result on screen.
print("Removed", before - len(df), "rows,", len(df), "left")

Removed 34335 rows, 1033036 left


In [14]:
# Remember how many rows we have before this step.
before = len(df)

# Keep only the rows where the price is above 0, and making it an independent table
df = df[df["price"] > 0].copy()

steps.append(["Remove price of 0 or less", before - len(df), len(df)])

# Show how many prices have we deleted
print("Removed", before - len(df), "rows,", len(df), "left")

Removed 6019 rows, 1027017 left


In [15]:
# Taking returns into consideration by giving them a negative number.
df["revenue"] = (df["quantity"] * df["price"]).round(2)

# Defining returns
df["is_return"] = df["quantity"] < 0

# Defining the codes based on previous cells that are not products
not_product_codes = ["POST", "DOT", "M", "C2", "C3", "D", "S", "BANK CHARGES",
                     "ADJUST", "ADJUST2", "B", "AMAZONFEE", "CRUK", "TEST001", "TEST002"]

codes = df["stock_code"].str.upper()

# Defining the products and separing them from voucher codes and other doubted codes
df["is_product"] = ~(codes.isin(not_product_codes) | codes.str.startswith("GIFT"))

# Defining sales with a defined buyer (they have customer_id)
df["has_customer_id"] = df["customer_id"].notna()

# Changing customer_id type to Int64 so it is read as a whole number
df["customer_id"] = df["customer_id"].astype("Int64")

# Creating a new column so we clarify the month to classify sales after
df["month"] = df["invoice_date"].dt.to_period("M").dt.to_timestamp()

In [16]:
# Count how many rows are products and how many are not
print(df["is_product"].value_counts())

# Counting codes treated as products that do NOT start with 5 digits, since they have descriptions that match as products.
print(df.loc[df["is_product"] & ~df["stock_code"].str.match(r"^\d{5}"), "stock_code"].value_counts())

is_product
True     1021272
False       5745
Name: count, dtype: int64
stock_code
DCGS0058     30
DCGSSGIRL    23
DCGSSBOY     21
PADS         18
DCGS0076     14
DCGS0003     13
DCGS0069      5
DCGS0004      4
DCGS0072      3
DCGS0068      2
DCGS0070      2
DCGS0066N     2
SP1002        2
DCGS0044      1
DCGS0037      1
DCGS0041      1
DCGS0075      1
DCGS0062      1
Name: count, dtype: int64


In [17]:
# Result table to see the removed rows
steps_table = pd.DataFrame(steps, columns=["step", "rows_removed", "rows_left"])

print(steps_table)

                        step  rows_removed  rows_left
0        Start (both sheets)             0    1067371
1      Remove duplicate rows         34335    1033036
2  Remove price of 0 or less          6019    1027017


# Check and Saves

In [20]:
# Looking if theres duplicates left
print("Duplicates left:", df.duplicated().sum())

# Looking for incoherent prices (negative)
print("Prices of 0 or less left:", (df["price"] <= 0).sum())

# Checking the total revenue
print("Total revenue:", round(df["revenue"].sum(), 2))

# Checking the combination of cancelated orders or negative quantity orders
print(pd.crosstab(df["invoice"].str.startswith("C"), df["quantity"] < 0,
                  rownames=["starts with C"], colnames=["negative quantity"]))

# Looking for sales with a buyer (customer_id)
with_id = df.loc[df["has_customer_id"], "revenue"].sum()

# Looking how much of the total revenue has a customer_id and comparing
print("Revenue with a customer number:", round(with_id / df["revenue"].sum() * 100, 1), "%")

Duplicates left: 0
Prices of 0 or less left: 0
Total revenue: 19014209.82
negative quantity    False  True 
starts with C                    
False              1007913      0
True                     1  19103
Revenue with a customer number: 85.7 %


In [21]:
# Looking for the odd row, A positive quantity and a cancelation at the same time
print(df[df["invoice"].str.startswith("C") & (df["quantity"] > 0)].T)

                               76799
invoice                      C496350
stock_code                         M
description                   Manual
quantity                           1
invoice_date     2010-02-01 08:24:00
price                         373.57
customer_id                     <NA>
country               United Kingdom
revenue                       373.57
is_return                      False
is_product                     False
has_customer_id                False
month            2010-02-01 00:00:00


19,103 lines have a "C", they are a cancelation invoice and a negative quantity, no line has a negative quantity without a C. But, there is one line (invoice C496350, manual entry, +£373.57, no customer) has a C invoice with a positive quantity. We decided to keep it because its size is too small to make a diference, and it is not a product

In [22]:
# Saving the clean table into our drive
df.to_parquet(folder + "transactions_clean.parquet", index=False)

# Saving the steps to help documenting the process after
steps_table.to_csv(folder + "steps_table.csv", index=False)

# Checking the number of rows saved
print("Saved", len(df), "rows")

Saved 1027017 rows
